**Sampler Demo - Running on IBM Quantum Hardware.**

This notebook builds a $\Psi^+$ Bell state, submits it to a **real** quantum
processor through IBM Quantum's cloud runtime, and retrieves the actual
measured bitstrings from the chip.

**This notebook submits a real job to your IBM Quantum account.**
It requires saved credentials (see `qiskit_save_credential.ipynb`) and it
consumes runtime from your account. Unlike the simulator version, Cell 03
can block for minutes or hours depending on the queue.

Any outputs saved below are from a previous run on one particular backend.
Because `least_busy()` picks whichever device is free at the time, your run
will use a different machine, a different job ID, and different counts.
Rerun the notebook to get your own results.

---
**Cell 01 - Build the Psi+ Bell state circuit.**

A Hadamard gate on qubit 0 followed by a CNOT (control 0, target 1) produces
the $\Psi^+$ Bell state $\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle)$.

`measure_all()` adds a classical register and measures every qubit into it.
The Sampler reports what those classical bits held after each shot, so
without measurements there would be nothing to report.

This is the idealized *logical* circuit. It assumes perfect gates and
arbitrary qubit connectivity - real hardware has neither, which is why
Cell 02 must transpile it.

In [ ]:
"""sampler_hardware.ipynb"""

# Cell 01 - Create a Psi+ Bell State (|00> + |11>)

from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

---
**Cell 02 - Select the least busy backend and submit.**

`QiskitRuntimeService(channel="ibm_cloud", instance=...)` loads the saved
credentials from `~/.qiskit/qiskit-ibm.json` and opens an authenticated
session.

**Why name the instance?**
Your API token identifies your *account*. The *instance* is the IBM Cloud
service instance that actually runs the job, and this course uses
`QIS101 - Open`.

If you do not name an instance, Qiskit picks one for you based on what is
saved in your credential file. That works, but nothing in the notebook tells
you which instance you got. Naming it means the code states plainly where the
job runs, and it behaves the same on any machine regardless of what is saved
there. If the name is wrong you get an immediate error rather than a silent
fallback to somewhere unexpected.

`least_busy(simulator=False, operational=True)` asks IBM which of the real
devices on your instance currently has the shortest job queue:

- `simulator=False` excludes cloud simulators, so we get actual hardware.
- `operational=True` excludes devices that are offline or in maintenance.

Because it picks whichever device is least busy right now, **you will not
always get the same backend**, and different devices have different qubit
counts, native gates, connectivity, and error rates. That is why the printed
backend name matters: it tells you which machine produced your counts.

The pass manager then rewrites the circuit into that specific device's native
gate set and maps our two logical qubits onto physical qubits that are
actually wired together on the chip. The result is the ISA circuit.

As in the simulator notebook, we let the runtime choose the number of shots.
Note that the hardware default is not the same as the simulator's - the
histogram in Cell 04 shows how many shots you actually got.

`sampler.run()` returns immediately with a `job` handle - the circuit has been
queued but has not yet run. **Save the printed job ID.** With it you can
retrieve your results later via `service.job(job_id)` even if this notebook
session ends before the hardware gets to your job.

In [ ]:
# Cell 02 - Select the least busy backend and submit the job

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

# An IBM Cloud "instance" is the service instance that runs your job.
# Name it explicitly so this notebook does not depend on whichever instance
# happens to be saved as the default on your machine.
INSTANCE_NAME = "QIS101 - Open"

# Connect to IBM Cloud using credentials in ~/.qiskit/qiskit-ibm.json
service = QiskitRuntimeService(channel="ibm_cloud", instance=INSTANCE_NAME)
print(f"Using instance: {INSTANCE_NAME}")

# Select the least busy real device that is currently operational
backend = service.least_busy(simulator=False, operational=True)
config = backend.configuration()
print(
    f"{config.backend_name:15}: Qubits = {config.n_qubits}: Gates = {config.basis_gates}"
)
print(f"Pending jobs in this backend's queue: {backend.status().pending_jobs}")

# Transpile the circuit for this backend's native gate set and connectivity
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
display(isa_circuit.draw("mpl", idle_wires=False))

# Run the circuit, letting the runtime choose the number of shots
sampler = Sampler(backend)
job = sampler.run([isa_circuit])
print(f"\nJob ID: {job.job_id()}  (save this to retrieve results later)")
print(f"Status: {job.status()}")

---
**Cell 03 - Wait for the job and retrieve the actual counts.**

`job.result()` is a *blocking* call: it waits until your job reaches the front
of the queue, runs on the chip, and returns its data. Depending on how busy
the device is this can take anywhere from seconds to hours.
The cell will appear to hang while it waits - that is normal.

If you would rather not wait, interrupt the cell and come back later with:

```python
job = service.job("paste-your-job-id-here")
```

As in the simulator notebook, `measure_all()` names the classical register
`meas`, so the bitstrings live in `pub_result.data.meas`, and `get_counts()`
tallies them into a dictionary of bitstring to shot count.

In [ ]:
# Cell 03 - Wait for the job to complete and retrieve the actual counts

# job.result() blocks until the hardware job finishes (may take minutes)
pub_result = job.result()[0]

# If you would rather not wait, interrupt the cell and come back here later with:
# job = service.job("paste-your-job-id-here")

# measure_all() names the classical register "meas"
counts = pub_result.data.meas.get_counts()
total_shots = sum(counts.values())

print(f"Backend: {config.backend_name}")
print(f"Total shots actually run: {total_shots}\n")
print(f"{'Bitstring':>10}{'Counts':>10}{'Percent':>10}")
for bitstring in sorted(counts):
    shots = counts[bitstring]
    print(f"{bitstring:>10}{shots:>10}{shots / total_shots:>10.2%}")

# The ideal Bell state never yields 01 or 10 - every such shot is an error
error_shots = counts.get("01", 0) + counts.get("10", 0)
print(
    f"\nShots that disagree (01 or 10): {error_shots} ({error_shots / total_shots:.2%})"
)
print("An ideal Bell state would give 0% - this is the hardware error rate.")

---
**Cell 04 - Plot the counts as a histogram.**

Two tall bars at `00` and `11` of roughly equal height, and two short bars at
`01` and `10`.

The tall bars are the entanglement: neither qubit has a definite value on its
own, yet the two always agree. The short bars are the part the ideal state
forbids entirely - every shot in them is a readout or gate error on a real
superconducting chip.

Compare this histogram to the one from `sampler_simulator.ipynb`.
The `FakeAlmadenV2` noise model is calibrated to mimic real hardware, so the
two plots should look broadly similar. Where they differ is instructive: the
fake backend is frozen at one past calibration snapshot of one device, while
your live backend has its own error rates that drift from day to day.

This is also the picture the Estimator hid from you. In
`estimator_hardware.ipynb` all of this collapsed into
$\langle ZZ \rangle \approx 1$. Same experiment, same machine, but here you
see every individual shot that went into that one number.

In [ ]:
# Cell 04 - Plot the counts as a histogram

from qiskit.visualization import plot_histogram

# plot_histogram builds and returns its own figure, so let the cell
# display that figure directly rather than calling plt.show()
plot_histogram(
    counts,
    color="steelblue",
    title=f"Psi+ Bell State Counts ({config.backend_name})",
)